# 메인 데이터셋

In [7]:
# 1. 필요 라이브러리 호출
import pandas as pd
import numpy as np
import requests
import time
import re
from sklearn.preprocessing import MinMaxScaler
import time
import math


print("라이브러리 호출 완료")

라이브러리 호출 완료


In [8]:
# 2. 메인 데이터셋(실거래가 및 단지정보) 불러오기
print("원본 데이터를 불러오는 중입니다...")

df_trade = pd.read_csv('아파트실거래가_2021_2025.csv', encoding='utf-8-sig', low_memory=False)
df_kapt = pd.read_excel('20260522_단지_기본정보.xlsx', skiprows=1)

print(f"원본 실거래가 데이터 개수: {len(df_trade):,}건")
print(f"원본 단지정보 데이터 개수: {len(df_kapt):,}건")

원본 데이터를 불러오는 중입니다...
원본 실거래가 데이터 개수: 12,454건
원본 단지정보 데이터 개수: 21,603건


In [9]:
# 3. 조인용 마스터키(주소) 정제
print("마스터키 정제를 시작합니다...")

# 1) 실거래가 주소 조합
df_trade['조인용_주소'] = df_trade['시군구'].astype(str).str.strip() + ' ' + df_trade['번지'].astype(str).str.strip()

# 2) 단지정보 주소 정제 (법정동주소에 포함된 단지명 제거)
def clean_kapt_address(row):
    addr = str(row['법정동주소']).strip()
    apt = str(row['단지명']).strip()
    if apt in addr:
        addr = addr.replace(apt, '').strip()
    return addr

df_kapt['조인용_주소'] = df_kapt.apply(clean_kapt_address, axis=1)

# 3) 다중 공백을 단일 공백으로 통일
df_trade['조인용_주소'] = df_trade['조인용_주소'].str.replace(r'\s+', ' ', regex=True)
df_kapt['조인용_주소'] = df_kapt['조인용_주소'].str.replace(r'\s+', ' ', regex=True)

# 4) 단지정보 데이터의 주소 중복 제거 (안전장치)
df_kapt = df_kapt.drop_duplicates(subset=['조인용_주소'], keep='first')

print("마스터키 정제 완료")

마스터키 정제를 시작합니다...
마스터키 정제 완료


In [10]:
# 4. 데이터 병합 (Left Join)
print("데이터를 병합하는 중입니다...")
df_main = pd.merge(df_trade, df_kapt, on='조인용_주소', how='left')

# 병합 시 중복으로 생성된 단지명 컬럼(단지명_x, 단지명_y) 깔끔하게 정리
if '단지명_x' in df_main.columns:
    df_main = df_main.rename(columns={'단지명_x': '단지명'})
    df_main = df_main.drop(columns=['단지명_y'], errors='ignore')

print("데이터 병합 및 컬럼 정리 완료")

데이터를 병합하는 중입니다...
데이터 병합 및 컬럼 정리 완료


In [11]:
# 5. 결과 확인 및 저장
print("[병합 결과 브리핑]")
print(f"원본 실거래가 데이터 개수: {len(df_trade):,}건")
print(f"병합 후 메인 데이터 개수: {len(df_main):,}건")
print("-" * 30)

unmatched_count = df_main['동수'].isna().sum()
matched_count = len(df_main) - unmatched_count

print(f"매칭된 데이터 개수: {matched_count:,}건")
print(f"누락된 데이터 개수: {unmatched_count:,}건")
print("-" * 30)

# 최종 파일 저장
df_main.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')
print("'final_dataset.csv' 파일로 저장이 완료되었습니다")

[병합 결과 브리핑]
원본 실거래가 데이터 개수: 12,454건
병합 후 메인 데이터 개수: 12,454건
------------------------------
매칭된 데이터 개수: 10,106건
누락된 데이터 개수: 2,348건
------------------------------
'final_dataset.csv' 파일로 저장이 완료되었습니다


# 카카오 지도 API

In [12]:
# 1. 데이터 불러오기 및 검색용 주소 자동 인식
print("데이터를 불러오고 검색용 주소를 세팅합니다...")
df_main = pd.read_csv('main_data.csv', encoding='utf-8-sig', low_memory=False)

if '조인용_주소' in df_main.columns:
    df_main['검색용_주소'] = df_main['조인용_주소']
elif '시군구' in df_main.columns and '번지' in df_main.columns:
    df_main['검색용_주소'] = df_main['시군구'].astype(str).str.strip() + ' ' + df_main['번지'].astype(str).str.strip()

# 2. 고유 주소만 추출 (중복 검색 방지하여 API 호출 낭비 최소화)
unique_addresses = pd.DataFrame(df_main['검색용_주소'].unique(), columns=['검색용_주소'])
print(f"카카오 API에 검색할 고유 아파트 주소 개수: {len(unique_addresses):,}개")

데이터를 불러오고 검색용 주소를 세팅합니다...
카카오 API에 검색할 고유 아파트 주소 개수: 181개


In [13]:
# 3. 카카오 로컬 API 지오코딩 함수 및 호출
REST_API_KEY = "02ce952b5588d51d2fb6034d03ae55dd" # 회원님의 API 키

def get_lat_lon(address):
    url = f"https://dapi.kakao.com/v2/local/search/address.json?query={address}"
    headers = {"Authorization": f"KakaoAK {REST_API_KEY}"}
    
    try:
        response = requests.get(url, headers=headers)
        result = response.json()
        
        if result['documents']: 
            match = result['documents'][0]
            # y가 위도, x가 경도
            return pd.Series([float(match['y']), float(match['x'])])
        else:
            return pd.Series([None, None])
            
    except Exception as e:
        return pd.Series([None, None])

print("카카오 API를 통해 위경도 좌표를 가져오는 중입니다...")
# API 호출 실행
unique_addresses[['아파트위도', '아파트경도']] = unique_addresses['검색용_주소'].apply(get_lat_lon)
print("위경도 추출 완료")

카카오 API를 통해 위경도 좌표를 가져오는 중입니다...
위경도 추출 완료


In [14]:
# 4. 원본 데이터에 위경도 병합 및 저장
df_final = pd.merge(df_main, unique_addresses, on='검색용_주소', how='left')

# 임시 컬럼 삭제
df_final = df_final.drop(columns=['검색용_주소'])

# 5. 결과 확인
missing_geo = df_final['아파트위도'].isna().sum()
print("[위경도 추출 결과 브리핑]")
print(f"원본 데이터 수: {len(df_main):,}건 / 병합 후 데이터 수: {len(df_final):,}건")
print(f"좌표 누락된 데이터: {missing_geo:,}건")

# 최종 저장
df_final.to_csv('main_data_with_geo.csv', index=False, encoding='utf-8-sig')
print("좌표가 포함된 최종 데이터가 'main_data_with_geo.csv'로 저장되었습니다!")

[위경도 추출 결과 브리핑]
원본 데이터 수: 12,454건 / 병합 후 데이터 수: 12,454건
좌표 누락된 데이터: 0건
좌표가 포함된 최종 데이터가 'main_data_with_geo.csv'로 저장되었습니다!


# 기본 세팅

In [15]:
print("메인 데이터를 불러옵니다...")
df = pd.read_csv('main_data_with_geo.csv', encoding='utf-8-sig', low_memory=False)

# 하버사인 거리 계산 공식
def calculate_haversine(lat1, lon1, lat2_array, lon2_array):
    R = 6371000.0
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2_array, lon2_array = np.radians(lat2_array), np.radians(lon2_array)
    dlat = lat2_array - lat1
    dlon = lon2_array - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2_array) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

print("기본 환경 및 거리 계산 함수 세팅 완료")

메인 데이터를 불러옵니다...
기본 환경 및 거리 계산 함수 세팅 완료


# 역세권 여부

In [16]:
print("지하철 역세권 여부 계산 중...")
df_subway = pd.read_csv('subway_geo_seongbuk.csv', encoding='utf-8-sig').dropna(subset=['역위도', '역경도'])
sub_lat, sub_lon = df_subway['역위도'].values, df_subway['역경도'].values

def get_subway_status(row):
    lat, lon = row['아파트위도'], row['아파트경도']
    if pd.isna(lat) or pd.isna(lon): return 0
    dist = calculate_haversine(lat, lon, sub_lat, sub_lon)
    return 1 if len(dist) > 0 and np.min(dist) <= 500 else 0

df['역세권여부'] = df.apply(get_subway_status, axis=1)
print("역세권여부(반경 500m 이내 지하철 유무) 계산 완료")

지하철 역세권 여부 계산 중...
역세권여부(반경 500m 이내 지하철 유무) 계산 완료


# 버스정류장 개수

In [17]:
print("반경 500m 버스정류장 개수 계산 중...")
df_bus = pd.read_excel('서울시 버스정류소 위치정보.xlsx').drop_duplicates(subset=['X좌표', 'Y좌표'])
bus_lat, bus_lon = df_bus['Y좌표'].values, df_bus['X좌표'].values

def get_bus_count(row):
    lat, lon = row['아파트위도'], row['아파트경도']
    if pd.isna(lat) or pd.isna(lon): return 0
    dist = calculate_haversine(lat, lon, bus_lat, bus_lon)
    return int(np.sum(dist <= 500))

df['반경500m_버스정류장'] = df.apply(get_bus_count, axis=1)
print("버스정류장 개수 계산 완료")

반경 500m 버스정류장 개수 계산 중...
버스정류장 개수 계산 완료


# 500m 내 편의시설 수 (공원+병원+상권)

In [18]:
print("반경 500m 편의시설(공원+병원+상권) 총합 계산 중...")

# 1. 3가지 시설 데이터 한 번에 로드 및 정제
df_park = pd.read_csv('성북구_공원.csv', encoding='utf-8-sig')
park_lat, park_lon = df_park['위도'].values, df_park['경도'].values

df_hosp = pd.read_csv('성북구_병원.csv', encoding='utf-8-sig')
hosp_lat, hosp_lon = df_hosp['위도'].values, df_hosp['경도'].values

df_smallbiz = pd.read_csv('성북구_소상공인.csv', encoding='utf-8-sig')
if '상권업종중분류명' in df_smallbiz.columns:
    keywords = '소매|커피|카페|약국|제과|베이커리|편의점|슈퍼|마트'
    df_smallbiz = df_smallbiz[df_smallbiz['상권업종중분류명'].str.contains(keywords, na=False, regex=True)]
biz_lat, biz_lon = df_smallbiz['위도'].values, df_smallbiz['경도'].values

# 2. 편의시설 총합을 한 번에 구하는 함수
def get_amenity_total(row):
    lat, lon = row['아파트위도'], row['아파트경도']
    if pd.isna(lat) or pd.isna(lon): return 0
    
    # 3가지 시설 각각 거리 계산 후 500m 이내 개수 세기
    park_cnt = np.sum(calculate_haversine(lat, lon, park_lat, park_lon) <= 500)
    hosp_cnt = np.sum(calculate_haversine(lat, lon, hosp_lat, hosp_lon) <= 500)
    biz_cnt = np.sum(calculate_haversine(lat, lon, biz_lat, biz_lon) <= 500)
    
    return int(park_cnt + hosp_cnt + biz_cnt)

# 3. 데이터에 적용
df['500m내_편의시설_총합'] = df.apply(get_amenity_total, axis=1)
print("편의시설 총합 계산 완료")

# 4. 최종 완성본 저장
df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')

print(f"최종 컬럼 개수: {len(df.columns)}개")
print("'final_dataset.csv'가 저장되었습니다")

반경 500m 편의시설(공원+병원+상권) 총합 계산 중...
편의시설 총합 계산 완료
최종 컬럼 개수: 110개
'final_dataset.csv'가 저장되었습니다


# 편의시설 수 정규화 전처리

In [19]:
print("편의시설 수 정규화 전처리를 시작합니다...")

# 1. 파일 불러오기
df = pd.read_csv('final_dataset.csv', low_memory=False)

# 2. 정규화 적용
if '500m내_편의시설_총합' in df.columns:
    scaler = MinMaxScaler()
    df['500m내_편의시설_총합_norm'] = scaler.fit_transform(df[['500m내_편의시설_총합']])
    
    df['편의시설(정규화)'] = df['500m내_편의시설_총합_norm']
    
    df = df.drop(columns=['500m내_편의시설_총합_norm'])
    
    print("편의시설 수치 0~1 사이로 스케일링 완료")

# 3. 덮어쓰기 저장
df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')

# 4. 결과 요약 출력
if '편의시설(정규화)' in df.columns:
    print(f"\n저장 완료: 데이터 크기 {df.shape}")
    print("\n[편의시설(정규화) 요약 통계]")
    print(df['편의시설(정규화)'].describe().round(3))
    print("\n[데이터 미리보기]")
    print(df[['500m내_편의시설_총합', '편의시설(정규화)']].head())

편의시설 수 정규화 전처리를 시작합니다...
편의시설 수치 0~1 사이로 스케일링 완료

저장 완료: 데이터 크기 (12454, 111)

[편의시설(정규화) 요약 통계]
count    12454.000
mean         0.306
std          0.196
min          0.000
25%          0.148
50%          0.283
75%          0.393
max          1.000
Name: 편의시설(정규화), dtype: float64

[데이터 미리보기]
   500m내_편의시설_총합  편의시설(정규화)
0            412   0.073993
1           3444   0.809558
2           1276   0.283600
3           1222   0.270500
4           1445   0.324600


# 데이터 불러오기

In [20]:
print("변수 전처리 및 데이터셋 업데이트를 시작합니다...")

# 1. 원본 데이터 불러오기
df = pd.read_csv('final_dataset.csv', low_memory=False)
initial_col_count = len(df.columns)

print(f"데이터 불러오기 완료 (현재 컬럼 개수: {initial_col_count}개)")

변수 전처리 및 데이터셋 업데이트를 시작합니다...
데이터 불러오기 완료 (현재 컬럼 개수: 111개)


# 전용면적

In [21]:
print("'전용면적' 변수 생성 중...")

if '전용면적(㎡)' in df.columns:
    df['전용면적'] = df['전용면적(㎡)']
    print("'전용면적' 생성 완료")

'전용면적' 변수 생성 중...
'전용면적' 생성 완료


# 건물나이

In [22]:
print("'건물나이' 변수 생성 중...")

if '계약년월' in df.columns and '건축년도' in df.columns:
    # 계약년월(예: 202405)에서 앞 4자리 연도만 잘라내서 건축년도를 뺌
    df['건물나이'] = df['계약년월'].astype(str).str[:4].astype(int) - df['건축년도']
    print("'건물나이' 생성 완료")

'건물나이' 변수 생성 중...
'건물나이' 생성 완료


# 세대당 주차대수

In [23]:
print("'세대당_주차대수' 변수 생성 중...")

if '총주차대수' in df.columns and '세대수' in df.columns:
    # 0으로 나누는 에러 방지 후 주차대수 계산
    df['세대당_주차대수'] = df['총주차대수'] / df['세대수'].replace(0, np.nan)
    df['세대당_주차대수'] = df['세대당_주차대수'].fillna(0) 
    print("'세대당_주차대수' 생성 완료")

'세대당_주차대수' 변수 생성 중...
'세대당_주차대수' 생성 완료


# 거래금액 (만원)

In [24]:
print("'거래금액_만원' 숫자형 정제 중...")

if '거래금액(만원)' in df.columns:
    # 쉼표가 섞인 문자열이면 쉼표 제거 후 float으로, 아니면 그대로 복사
    if df['거래금액(만원)'].dtype == 'object':
        df['거래금액_만원'] = df['거래금액(만원)'].astype(str).str.replace(',', '').astype(float)
    else:
        df['거래금액_만원'] = df['거래금액(만원)']
    print("'거래금액_만원' 정제 완료")

'거래금액_만원' 숫자형 정제 중...
'거래금액_만원' 정제 완료


# 데이터 저장

In [25]:
print("최종 데이터 저장 및 브리핑을 진행합니다...")

# 기존 파일에 안전하게 업데이트(덮어쓰기)
df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')

# 방금 만든 변수들만 모아서 잘 들어갔는지 화면에 출력
added_cols = ['전용면적', '건물나이', '세대당_주차대수', '거래금액_만원']
valid_added_cols = [c for c in added_cols if c in df.columns]

print("\n [추가된 변수 미리보기]")
display(df[valid_added_cols].head())

# 늘어난 컬럼 개수 계산
final_col_count = len(df.columns)
print("\n" + "=" * 50)
print(f"원본 컬럼 개수: {initial_col_count}개")
print(f"업데이트 후 최종 컬럼 개수: {final_col_count}개 (+{final_col_count - initial_col_count}개 추가됨)")
print("=" * 50)
print("변수별 업데이트가 'final_dataset.csv'에 완벽하게 저장되었습니다!")

최종 데이터 저장 및 브리핑을 진행합니다...

 [추가된 변수 미리보기]


,전용면적,건물나이,세대당_주차대수,거래금액_만원
0,59.8800,16,1.198376,60000.0
1,59.9400,22,0.981693,65000.0
2,15.2144,2,0.511706,11550.0
3,84.9600,21,0.871203,76000.0
4,59.5800,23,1.222703,71000.0



원본 컬럼 개수: 111개
업데이트 후 최종 컬럼 개수: 115개 (+4개 추가됨)
변수별 업데이트가 'final_dataset.csv'에 완벽하게 저장되었습니다!


# 기준 금리

In [26]:
print("'기준금리' 변수 병합 중...")


if '계약년월' in df.columns:
    df['계약년'] = df['계약년월'].astype(str).str[:4].astype(int)
    
    try:
        df_interest = pd.read_csv('interest_clean.csv')
        # 계약년을 기준으로 병합 (Left Join)
        df = pd.merge(df, df_interest, on='계약년', how='left')
        
        # 빈칸이 생기면 이전 데이터로 채우거나 0으로 처리 (최신 판다스 문법 적용)
        df['기준금리'] = df['기준금리'].ffill().fillna(0)
        print("'기준금리' 병합 완료")
    except FileNotFoundError:
        print("'interest_clean.csv' 파일이 없어 기준금리를 0으로 임시 채웁니다.")
        df['기준금리'] = 0

'기준금리' 변수 병합 중...
'기준금리' 병합 완료


# 고층 여부

In [27]:
print("'고층여부' 변수 생성 중...")

if '층' in df.columns:
    # 층수를 온전한 숫자로 변환
    df['층'] = pd.to_numeric(df['층'], errors='coerce').fillna(0)
    
    # 15층 이상이면 1, 아니면 0
    df['고층여부'] = (df['층'] >= 15).astype(int)
    print("'층' 정제 및 '고층여부' 생성 완료")

'고층여부' 변수 생성 중...
'층' 정제 및 '고층여부' 생성 완료


# 브랜드 TOP20 여부

In [28]:
print("'브랜드_Top20_여부' 파생변수 생성 중...")

top_brands = '래미안|자이|푸르지오|이편한|e편한|힐스테이트|더샵|롯데캐슬|아이파크|SK|데시앙|센트레빌|포레나|꿈에그린|스위첸|어울림|리슈빌|하늘채|베르디움|호반|우미'
brand_mask = pd.Series(False, index=df.index)

if '시공사' in df.columns:
    brand_mask = brand_mask | df['시공사'].astype(str).str.contains(top_brands, na=False, regex=True)
if '단지명' in df.columns:
    brand_mask = brand_mask | df['단지명'].astype(str).str.contains(top_brands, na=False, regex=True)

df['브랜드_Top20_여부'] = brand_mask.astype(int)
print("'브랜드_Top20_여부' 생성 완료")

'브랜드_Top20_여부' 파생변수 생성 중...
'브랜드_Top20_여부' 생성 완료


# 데이터 저장

In [29]:
print("최종 데이터 저장 및 브리핑을 진행합니다...")

# 기존 파일에 안전하게 덮어쓰기
df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')
print("변수들이 성공적으로 업데이트 되었습니다.")

# 추가된 변수들만 모아서 표 형태로 예쁘게 출력
added_cols = ['기준금리', '층', '고층여부', '브랜드_Top20_여부']
if '계약년' in df.columns:
    added_cols.insert(0, '계약년')

valid_added_cols = [c for c in added_cols if c in df.columns]

print("\n [새로 업데이트된 변수 미리보기]")
display(df[valid_added_cols].head()) 

# 컬럼 증가량 브리핑
final_col_count = len(df.columns)
print("\n" + "=" * 50)
print(f"이전 컬럼 개수: {initial_col_count}개")
print(f"업데이트 후 최종 컬럼 개수: {final_col_count}개 (+{final_col_count - initial_col_count}개 추가됨)")
print("=" * 50)

최종 데이터 저장 및 브리핑을 진행합니다...
변수들이 성공적으로 업데이트 되었습니다.

 [새로 업데이트된 변수 미리보기]


,계약년,기준금리,층,고층여부,브랜드_Top20_여부
0,2021,1.0,6,0,0
1,2021,1.0,5,0,0
2,2021,1.0,12,0,0
3,2021,1.0,10,0,0
4,2021,1.0,11,0,0



이전 컬럼 개수: 111개
업데이트 후 최종 컬럼 개수: 119개 (+8개 추가됨)


# 고도

In [30]:
print(" 고도 데이터 추출을 시작합니다...")

# 1. 파일 불러오기
df = pd.read_csv('final_dataset.csv', low_memory=False)
initial_col_count = len(df.columns)

# 2. 고도 데이터 추출 함수
def get_elevation_batch(lat_list, lon_list):
    locations = "|".join([f"{lat},{lon}" for lat, lon in zip(lat_list, lon_list)])
    url = f"https://api.opentopodata.org/v1/srtm30m?locations={locations}"
    
    try:
        res = requests.get(url, timeout=15)
        res.raise_for_status()
        data = res.json()
        
        if "results" in data:
            return [r["elevation"] for r in data["results"]]
        else:
            return [0] * len(lat_list)
            
    except Exception as e:
        print(f"\n API 통신 에러 (0으로 임시 대체): {e}")
        return [0] * len(lat_list)

# 3. 아파트 위경도 바탕으로 고도 조회 및 추가
if '아파트위도' in df.columns and '아파트경도' in df.columns:
    elevations = []
    batch_size = 100
    total_rows = len(df)
    
    print(f"총 {total_rows}개 아파트 좌표의 고도를 조회합니다.")
    
    for i in range(0, total_rows, batch_size):
        batch_lat = df["아파트위도"].iloc[i:i+batch_size].tolist()
        batch_lon = df["아파트경도"].iloc[i:i+batch_size].tolist()
        
        elevations.extend(get_elevation_batch(batch_lat, batch_lon))
        
        time.sleep(1.2) 
        
        if (i + batch_size) % 1000 == 0 or (i + batch_size) >= total_rows:
            print(f"▶ {min(i + batch_size, total_rows)} / {total_rows} 개 고도 조회 완료...")
            
    df['고도'] = np.round(elevations, 1)

# 4. 덮어쓰기 저장 및 결과 출력
df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')

print("\n 고도 데이터 추가가 끝났습니다")

added_cols = ['아파트위도', '아파트경도', '고도']
available_cols = [c for c in added_cols if c in df.columns]
print(df[available_cols].head())

final_col_count = len(df.columns)
print("-" * 50)
print(f"이전 컬럼 개수: {initial_col_count}개")
print(f"업데이트 후 최종 컬럼 개수: {final_col_count}개 (+{final_col_count - initial_col_count}개 추가됨)")
print("-" * 50)

 고도 데이터 추출을 시작합니다...
총 12454개 아파트 좌표의 고도를 조회합니다.
▶ 1000 / 12454 개 고도 조회 완료...
▶ 2000 / 12454 개 고도 조회 완료...
▶ 3000 / 12454 개 고도 조회 완료...
▶ 4000 / 12454 개 고도 조회 완료...
▶ 5000 / 12454 개 고도 조회 완료...
▶ 6000 / 12454 개 고도 조회 완료...
▶ 7000 / 12454 개 고도 조회 완료...
▶ 8000 / 12454 개 고도 조회 완료...
▶ 9000 / 12454 개 고도 조회 완료...
▶ 10000 / 12454 개 고도 조회 완료...
▶ 11000 / 12454 개 고도 조회 완료...
▶ 12000 / 12454 개 고도 조회 완료...
▶ 12454 / 12454 개 고도 조회 완료...

 고도 데이터 추가가 끝났습니다
       아파트위도       아파트경도     고도
0  37.618488  127.007298  142.0
1  37.590335  127.012681   41.0
2  37.613647  127.061019   24.0
3  37.600962  127.015704   67.0
4  37.594178  127.010321  116.0
--------------------------------------------------
이전 컬럼 개수: 119개
업데이트 후 최종 컬럼 개수: 120개 (+1개 추가됨)
--------------------------------------------------


# 경사도

In [31]:
print("아파트 주변 4방향 고도 조회 및 경사도 계산을 시작합니다...")

# 1. 파일 불러오기
df = pd.read_csv('final_dataset.csv', low_memory=False)

# 2. 거리 및 위경도 오프셋 설정 (111m 기준)
LAT_OFFSET = 0.001
LON_OFFSET = 0.00125
DISTANCE_M = 111.0

def get_elevations(locations_str):
    url = f"https://api.opentopodata.org/v1/srtm30m?locations={locations_str}"
    try:
        res = requests.get(url, timeout=15)
        res.raise_for_status()
        data = res.json()
        if "results" in data:
            return [r["elevation"] for r in data["results"]]
    except Exception as e:
        print(f"API 통신 에러: {e}")
    return None

slopes = []

if '아파트위도' in df.columns and '아파트경도' in df.columns:
    total_rows = len(df)
    batch_size = 20
    
    print(f"총 {total_rows}개 아파트의 경사도를 계산합니다.")
    
    for i in range(0, total_rows, batch_size):
        batch_df = df.iloc[i:i+batch_size]
        locations_list = []
        
        for _, row in batch_df.iterrows():
            lat = row['아파트위도']
            lon = row['아파트경도']
            
            locations_list.extend([
                f"{lat + LAT_OFFSET},{lon}",
                f"{lat - LAT_OFFSET},{lon}",
                f"{lat},{lon + LON_OFFSET}",
                f"{lat},{lon - LON_OFFSET}"
            ])
            
        locations_str = "|".join(locations_list)
        elevations = get_elevations(locations_str)
        
        if elevations and len(elevations) == len(batch_df) * 4:
            for j in range(len(batch_df)):
                n_elev = elevations[j*4]
                s_elev = elevations[j*4 + 1]
                e_elev = elevations[j*4 + 2]
                w_elev = elevations[j*4 + 3]

                du = (e_elev - w_elev) / (2 * DISTANCE_M)

                dv = (n_elev - s_elev) / (2 * DISTANCE_M)
       
                gradient_magnitude = math.sqrt(du**2 + dv**2)
           
                slope_angle = math.degrees(math.atan(gradient_magnitude))
                slopes.append(round(slope_angle, 2))
        else:
            slopes.extend([0.0] * len(batch_df))
            
        time.sleep(1.2)
        
        if (i + batch_size) % 100 == 0 or (i + batch_size) >= total_rows:
            print(f" {min(i + batch_size, total_rows)} / {total_rows} 개 아파트 경사도 계산 완료...")

    df['경사도'] = slopes
    df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')
    
    print("\n벡터 기반 경사도 계산 완료")
    print(df[['단지명', '경사도']].head())


아파트 주변 4방향 고도 조회 및 경사도 계산을 시작합니다...
총 12454개 아파트의 경사도를 계산합니다.
 100 / 12454 개 아파트 경사도 계산 완료...
 200 / 12454 개 아파트 경사도 계산 완료...
 300 / 12454 개 아파트 경사도 계산 완료...
 400 / 12454 개 아파트 경사도 계산 완료...
 500 / 12454 개 아파트 경사도 계산 완료...
 600 / 12454 개 아파트 경사도 계산 완료...
 700 / 12454 개 아파트 경사도 계산 완료...
 800 / 12454 개 아파트 경사도 계산 완료...
 900 / 12454 개 아파트 경사도 계산 완료...
 1000 / 12454 개 아파트 경사도 계산 완료...
 1100 / 12454 개 아파트 경사도 계산 완료...
 1200 / 12454 개 아파트 경사도 계산 완료...
 1300 / 12454 개 아파트 경사도 계산 완료...
 1400 / 12454 개 아파트 경사도 계산 완료...
 1500 / 12454 개 아파트 경사도 계산 완료...
 1600 / 12454 개 아파트 경사도 계산 완료...
 1700 / 12454 개 아파트 경사도 계산 완료...
 1800 / 12454 개 아파트 경사도 계산 완료...
 1900 / 12454 개 아파트 경사도 계산 완료...
 2000 / 12454 개 아파트 경사도 계산 완료...
 2100 / 12454 개 아파트 경사도 계산 완료...
 2200 / 12454 개 아파트 경사도 계산 완료...
 2300 / 12454 개 아파트 경사도 계산 완료...
 2400 / 12454 개 아파트 경사도 계산 완료...
 2500 / 12454 개 아파트 경사도 계산 완료...
 2600 / 12454 개 아파트 경사도 계산 완료...
 2700 / 12454 개 아파트 경사도 계산 완료...
 2800 / 12454 개 아파트 경사도 계산 완료...
 2900 / 12454 개 아파트 경사도